# 📄 Contracts OCR Pipeline — Arabic Legal Document Processing

**Part 1: OCR Engineer & Arabic Text Preparation**

هذا الـ Notebook يقوم ببناء Pipeline كامل لتحويل المستندات القانونية العربية
(PDF أو صور) إلى نص عربي منظم ومقسّم إلى بنود (Clauses).



In [ ]:
# 1. تحديث حزم النظام وتثبيت أدوات OCR المعالجة للغة العربية و Poppler
!apt-get -qq update
!apt-get -qq install -y poppler-utils tesseract-ocr tesseract-ocr-ara > /dev/null 2>&1

# 2. تثبيت مكتبات معالجة الملفات والصور ومكتبة Groq
!pip -q install PyMuPDF python-docx pdf2image pillow pytesseract opencv-python numpy groq

print("✅ تم تثبيت جميع المتطلبات وأدوات معالجة الصور بنجاح!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ تم تثبيت جميع المتطلبات وأدوات معالجة الصور بنجاح!


## 2️⃣ Imports

In [ ]:
import os
import re
import io
import json
import csv
import glob
import time
import unicodedata
import traceback
from datetime import datetime

import fitz  # PyMuPDF
import docx
from PIL import Image
from pdf2image import convert_from_path
import cv2
import numpy as np
import pytesseract  # التعرف البصري على الحروف

# 1. التحقق من مكتبة Groq
try:
    from groq import Groq
    HAS_GROQ = True
except ImportError:
    HAS_GROQ = False
    print("⚠️ مكتبة groq غير مثبتة، يمكنك تثبيتها عبر: !pip install groq")

# 2. التحقق من بيئة Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("⚠️ لست داخل Google Colab — سيتم تعطيل خطوات الرفع/التحميل التلقائية.")

# 3. جلب وتثبيت مفتاح GROQ_API_KEY بأمان
if IN_COLAB:
    try:
        from google.colab import userdata
        groq_key = userdata.get('GROQ_API_KEY')
        if groq_key and isinstance(groq_key, str) and groq_key.strip():
            os.environ["GROQ_API_KEY"] = groq_key.strip()
            print("🔑 تم تحميل GROQ_API_KEY بنجاح من Colab Secrets.")
        else:
            print("⚠️ المفتاح في Colab Secrets غير مفعل أو فارغ.")
    except Exception as e:
        print(f"⚠️ يتعذر جلب المفتاح تلقائياً من Secrets: {e}")

# التحقق النهائي من توفر المفتاح في البيئة
if not os.getenv("GROQ_API_KEY"):
    print("💡 تنبيه: لم يتم العثور على GROQ_API_KEY، سيتم الاعتماد على OCR المحلي فقط إن لزم الأمر.")

print("✅ تم استيراد جميع المكتبات وتجهيز البيئة بنجاح!")

🔑 تم تحميل GROQ_API_KEY بنجاح من Colab Secrets.
✅ تم استيراد جميع المكتبات وتجهيز البيئة بنجاح!


## 3️⃣ Upload File — رفع المستند



In [ ]:
SUPPORTED_EXTENSIONS = {
    ".pdf",
    ".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tiff",
    ".docx", ".doc",
    ".txt", ".csv", ".json"
}

uploaded_file_path = None

# التأكد السريع من وجود المفتاح في بيئة العمل
if not os.environ.get("GROQ_API_KEY"):
    if IN_COLAB:
        try:
            from google.colab import userdata
            groq_key = userdata.get('GROQ_API_KEY')
            if groq_key and isinstance(groq_key, str) and groq_key.strip():
                os.environ["GROQ_API_KEY"] = groq_key.strip()
                print("🔑 تم تحميل GROQ_API_KEY بنجاح.")
        except Exception as e:
            print(f"⚠️ تعذر جلب المفتاح: {e}")

# رفع الملف والتحقق من امتداده
if IN_COLAB:
    print("📤 من فضلك ارفع المستند (PDF, Word, صورة, أو ملف نصي)...")
    uploaded = files.upload()
    if uploaded:
        raw_name = list(uploaded.keys())[0]
        uploaded_file_path = raw_name.strip()

        ext = os.path.splitext(uploaded_file_path)[1].lower()
        if ext not in SUPPORTED_EXTENSIONS:
            raise ValueError(f"❌ نوع الملف غير مدعوم: {ext}. الأنواع المدعومة هي:\n{SUPPORTED_EXTENSIONS}")

        print(f"✅ تم رفع الملف بنجاح: {uploaded_file_path}")
    else:
        print("⚠️ لم يتم رفع أي ملف.")
else:
    print("💡 تنبيه: يرجى تحديد مسار الملف يدوياً في المتغير uploaded_file_path عند التشغيل المحلي.")

📤 من فضلك ارفع المستند (PDF, Word, صورة, أو ملف نصي)...


Saving عقد بيع الدور الثانى با رقم لعمارة 15.pdf to عقد بيع الدور الثانى با رقم لعمارة 15 (1).pdf
✅ تم رفع الملف بنجاح: عقد بيع الدور الثانى با رقم لعمارة 15 (1).pdf


## 4️⃣ Detect File Type — تحديد نوع الملف

In [ ]:
def detect_file_type(file_path):
    """يحدد نوع الملف: pdf, image, word, أو text ويتحقق من أنه مدعوم."""
    if not file_path:
        raise FileNotFoundError("لم يتم تحديد مسار الملف.")

    file_path = str(file_path).strip()
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"الملف غير موجود: {file_path}")

    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        return "pdf"
    elif ext in {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tiff"}:
        return "image"
    elif ext in {".docx", ".doc"}:
        return "word"
    elif ext in {".txt", ".csv", ".json"}:
        return "text"
    else:
        raise ValueError(f"نوع ملف غير مدعوم: {ext}")

if 'uploaded_file_path' in locals() and uploaded_file_path:
    file_type = detect_file_type(uploaded_file_path)
    print(f"📄 نوع الملف المكتشف: {file_type}")

📄 نوع الملف المكتشف: pdf


## 5️⃣ Text Extraction / OCR

- إذا كان الـ PDF يحتوي على نص قابل للاستخراج (text layer)، نستخدم استخراج النص مباشرة (أسرع وأدق).
- إذا كان الـ PDF ممسوحًا ضوئيًا (scanned) بدون نص، أو كان الملف صورة، نستخدم Tesseract OCR مع دعم اللغة العربية.
- يتم الحفاظ على ترتيب النص حسب ترتيب الصفحات وظهوره في المستند، ومعالجة كل صفحة بشكل مستقل بحيث لا يوقف خطأ في صفحة واحدة باقي المعالجة.


In [ ]:
import base64

# ==========================================
# 0. دالة التعرف على نوع الملفات
# ==========================================
def detect_file_type(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        return "pdf"
    elif ext in {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tiff"}:
        return "image"
    elif ext in {".docx", ".doc"}:
        return "word"
    elif ext in {".txt", ".csv", ".json"}:
        return "text"
    else:
        return "unknown"

# ==========================================
# 1. تهيئة عميل Groq API لتصحيح وتنسيق النصوص
# ==========================================
def get_groq_client():
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key or not api_key.startswith("gsk_"):
        raise ValueError("❌ لم يتم العثور على GROQ_API_KEY صالح في البيئة أو Colab Secrets.")
    return Groq(api_key=api_key)

def refine_text_with_groq(raw_text):
    """إصلاح تشابك الحروف وتنقية شوائب الـ OCR عبر Groq مع الحفاظ التام على محتوى العقد"""
    if not raw_text or not raw_text.strip():
        return ""

    # قائمة بأسماء النماذج الرسمية المتاحة على Groq
    GROQ_TEXT_MODELS = [
        "llama-3.3-70b-versatile",
        "llama3-70b-8192",
        "llama-3.1-8b-instant"
    ]

    prompt = f"""أنت خبير تدقيق نصوص قانونية ومستخرجات OCR باللغة العربية.
المطلوب منك إعادة ترتيب وتنظيف النص التالي المأخوذ من عقد قانوني:

التعليمات الصارمة:
1. قم بإصلاح الكلمات الملتصقة والمشوهة وإضافة المسافات الصحيحة بين الكلمات.
2. احذف شوائب الـ OCR والرموز الإنجليزية غير المفهومة (مثل Goll, cL tia, yi) وأرقام الصفحات المتداخلة وسط السطور.
3. رتب البنود والفقرات بتسلسلها المنطقي الصحيح (تمهيد، البند الأول، البند الثاني... إلخ).
4. حافظ على جميع الأسماء، الأرقام، التواريخ، والمبالغ المالية كما هي دون أي تعديل أو حذف.
5. لا تضف أي مقدمة أو خاتمة أو شرح من عندك، اكتب النص النهائي المنقح فقط.

النص الخام:
{raw_text}"""

    try:
        client = get_groq_client()
        for model_name in GROQ_TEXT_MODELS:
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.1
                )
                result = response.choices[0].message.content.strip()
                if result:
                    return result
            except Exception:
                continue
    except Exception as e:
        print(f"⚠️ تنبيه: تعذر الاتصال بـ Groq ({e})، سيتم استرجاع النص الخام.")

    return raw_text

# ==========================================
# 2. تحسين صورة الصفحة قبل الـ OCR (Pre-processing)
# ==========================================
def preprocess_image_for_ocr(pil_img):
    """تحسين تباين الصورة وإزالة الضوضاء لرفع دقة Tesseract"""
    cv_img = np.array(pil_img.convert('RGB'))
    gray = cv2.cvtColor(cv_img, cv2.COLOR_RGB2GRAY)

    # تحسين التباين
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    # تحويل الصورة إلى PIL
    return Image.fromarray(enhanced)

# ==========================================
# 3. استخراج نصوص PDF باستخدام Tesseract + Groq
# ==========================================
def extract_pdf_direct(pdf_path):
    page_texts = []
    try:
        # تحويل PDF إلى صور عالية الجودة
        images = convert_from_path(pdf_path, dpi=300)
        print(f"🔎 جاري معالجة {len(images)} صفحة بواسطة Tesseract OCR + Groq...")

        # psm 3 أحياناً أفضل للوثائق الكاملة، و psm 6 للكتل النصية
        custom_config = r'--oem 3 --psm 3 -l ara+eng'

        for i, img in enumerate(images):
            # 1. تحسين الصورة
            processed_img = preprocess_image_for_ocr(img)

            # 2. التعرف البصري عبر Tesseract
            raw_text = pytesseract.image_to_string(processed_img, config=custom_config)

            # 3. إصلاح وتنسيق النص عبر Llama
            corrected_text = refine_text_with_groq(raw_text)

            page_texts.append(corrected_text)
            print(f"   └─ ✅ تم استخراج وتنقية الصفحة {i + 1}/{len(images)} بنجاح")

        return page_texts

    except Exception as e:
        print(f"❌ خطأ أثناء استخراج ملف الـ PDF: {e}")
        return []

# ==========================================
# 4. باقي الدوال (صور / Word / نصي)
# ==========================================
def extract_image_direct(image_path):
    try:
        img = Image.open(image_path)
        processed_img = preprocess_image_for_ocr(img)
        custom_config = r'--oem 3 --psm 3 -l ara+eng'
        raw_text = pytesseract.image_to_string(processed_img, config=custom_config)
        corrected_text = refine_text_with_groq(raw_text)
        return [corrected_text] if corrected_text else [""]
    except Exception as e:
        print(f"❌ خطأ أثناء معالجة الصورة: {e}")
        return [""]

def extract_word_direct(docx_path):
    try:
        doc = docx.Document(docx_path)
        raw_text = "\n".join([p.text for p in doc.paragraphs if p.text.strip()])
        return [raw_text]
    except Exception as e:
        print(f"❌ خطأ في قراءة ملف Word: {e}")
        return [""]

def extract_text_file_direct(file_path):
    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            return [f.read()]
    except Exception as e:
        print(f"❌ خطأ في قراءة الملف النصي: {e}")
        return [""]

# ==========================================
# 5. الدالة الرئيسية
# ==========================================
def extract_document_text(file_path):
    file_type = detect_file_type(file_path)

    if file_type == "pdf":
        print("🔎 تشغيل معالجة ملف الـ PDF...")
        page_texts = extract_pdf_direct(file_path)
        method = "tesseract_groq_llama33"
    elif file_type == "image":
        print("🔎 جاري استخراج النصوص من الصورة...")
        page_texts = extract_image_direct(file_path)
        method = "tesseract_groq_image"
    elif file_type == "word":
        print("🔎 جاري قراءة مستند Word...")
        page_texts = extract_word_direct(file_path)
        method = "docx_direct"
    elif file_type == "text":
        print("🔎 جاري قراءة الملف النصي...")
        page_texts = extract_text_file_direct(file_path)
        method = "plain_text"
    else:
        raise ValueError(f"نوع ملف غير مدعوم: {file_type}")

    return page_texts, method, len(page_texts)

## 6️⃣ Raw Text — النص الخام

يتم دمج نصوص جميع الصفحات (بترتيب ظهورها) في `raw_text` كما استُخرجت تمامًا، بدون أي حذف أو إعادة صياغة للمحتوى القانوني.

In [ ]:
raw_text = ""
extraction_method = None
num_pages = 0
full_extracted_text = ""

# ==========================================
# 6. تشغيل الاستخراج وحساب الإحصائيات فقط
# ==========================================
if uploaded_file_path:
    # 1. استخراج النصوص
    page_texts, method, page_count = extract_document_text(uploaded_file_path)

    # دمج جميع الصفحات
    full_extracted_text = "\n\n--- صفحة جديدة ---\n\n".join([p for p in page_texts if p.strip()])
    total_chars = len(full_extracted_text.strip())
    total_words = len(full_extracted_text.split())
    file_name = os.path.basename(uploaded_file_path)

    # 2. التحقق وطباعة ملخص البيانات فقط
    if total_chars == 0:
        print("\n⚠️ تنبيه: اكتملت العملية لكن لم يتم العثور على نص قابل للقراءة داخل الملف.")
    else:
        print("\n✅ تم إستخراج النص بنجاح!")
        print(f"📁 اسم الملف: {file_name}")
        print(f"📄 عدد الصفحات: {page_count}")
        print(f"📝 عدد الكلمات: {total_words}")
        print(f"📏 عدد الأحرف: {total_chars}")
        print(f"📌 طريقة الاستخراج: {method}")

🔎 تشغيل معالجة ملف الـ PDF...
🔎 جاري معالجة 4 صفحة بواسطة Tesseract OCR + Groq...
   └─ ✅ تم استخراج وتنقية الصفحة 1/4 بنجاح
   └─ ✅ تم استخراج وتنقية الصفحة 2/4 بنجاح
   └─ ✅ تم استخراج وتنقية الصفحة 3/4 بنجاح
   └─ ✅ تم استخراج وتنقية الصفحة 4/4 بنجاح

✅ تم إستخراج النص بنجاح!
📁 اسم الملف: عقد بيع الدور الثانى با رقم لعمارة 15 (1).pdf
📄 عدد الصفحات: 4
📝 عدد الكلمات: 942
📏 عدد الأحرف: 5420
📌 طريقة الاستخراج: tesseract_groq_llama33


## 7️⃣ Arabic Text Cleaning — تنظيف النص العربي

نتعامل مع:
- المسافات الزائدة وتكرارها
- الأسطر الفارغة و line breaks غير الصحيحة
- Unicode normalization (NFC)
- Arabic Tatweel (ـ)
- علامات الترقيم المكررة/غير المنسقة
- نصوص/رموز غير ضرورية ناتجة عن OCR

**بدون** تغيير المعنى القانوني أو إعادة صياغة الجمل.


In [ ]:
import re

def fix_ocr_clause_headers(text):
    """تصحيح أخطاء OCR الشائعة في عناوين البنود والمواد وتنسيقها"""
    if not text:
        return ""

    # 1. إصلاح أخطاء الطباعة الحرفية في مسميات البنود
    text = re.sub(r'البند\s*العاشر', 'البند العاشر', text)
    text = re.sub(r'البند\s*الرا[بع]*', 'البند الرابع', text)
    text = re.sub(r'البند\s*السا[يئ]ع', 'البند السابع', text)
    text = re.sub(r'اليند', 'البند', text)

    # 2. دمج عناوين البنود المقسومة على سطرين (مثل: البند الثالث \n _عشر)
    text = re.sub(r'(البند\s+[أ-ي]+)\s*[\n\r_]+(عشر)', r'\1 \2', text)
    text = re.sub(r'(المادة\s+\w+)\s*[\n\r_]+(عشر)', r'\1 \2', text)

    # 3. معالجة التصاق عناوين البنود بالنصوص
    text = re.sub(r'(البند\s*[أ-ي]+)(?=[أ-ي])', r'\1 ', text)

    # 4. معالجة الأرقام المغلوطة والرموز داخل الأقواس (تصحيح تحويل 2 إلى " أو ؟)
    def clean_numeric_brackets(match):
        content = match.group(1)
        content = content.replace('"', '2').replace('؟', '2').replace("'", '')
        return f"({content.strip()})"

    text = re.sub(r'\(([^)\n]*[\d٠-٩"؟][^)\n]*)\)', clean_numeric_brackets, text)

    return text

def build_advanced_clause_pattern():
    """أنماط لاقتنص عناوين البنود والترقيمات القانونية"""
    words_pattern = (
        r"(?:الأول|الأولى|الثاني|الثانية|الثالث|الثالثة|الرابع|الرابعة|الخامس|الخامسة|"
        r"السادس|السادسة|السابع|السابعة|الثامن|الثامنة|التاسع|التاسعة|العاشر|العاشرة|"
        r"الحادي\s+عشر|الثاني\s+عشر|الثالث\s+عشر|الرابع\s+عشر|الخامس\s+عشر|السادس\s+عشر|"
        r"السابع\s+عشر|الثامن\s+عشر|التاسع\s+عشر|العشرون|الثلاثون|رقم\s*\d+|\d+)"
    )

    ordinals_pattern = (
        r"(?:أولاً|أولا|ثانياً|ثانيا|ثالثاً|ثالثا|رابعاً|رابعا|خامساً|خامسا|"
        r"سادساً|سادسا|سابعاً|سابعا|ثامناً|ثامنا|تاسعاً|تاسعا|عاشراً|عاشرا)"
    )

    patterns = [
        rf"(?:^|\n)\s*(البند\s+{words_pattern})\s*[:\-–—]?",
        rf"(?:^|\n)\s*(المادة\s+{words_pattern})\s*[:\-–—]?",
        rf"(?:^|\n)\s*({ordinals_pattern})\s*[:\-–—]?",
        r"(?:^|\n)\s*(\d{1,2}\s*[-.\)])\s*",
        r"(?:^|\n)\s*(\(\d{1,2}\))\s*"
    ]

    return re.compile("|".join(patterns), flags=re.MULTILINE)

def segment_legal_clauses_exact(text):
    if not text or not text.strip():
        return []

    # 1. تنظيف مسميات العناوين وتعديل شوائب الـ OCR
    prepared_text = fix_ocr_clause_headers(text)

    # 2. مطابقة كافّة البنود
    clause_regex = build_advanced_clause_pattern()
    matches = list(clause_regex.finditer(prepared_text))

    clauses = []

    if matches:
        # أ) التمهيد/الديباجة
        first_start = matches[0].start()
        preamble = prepared_text[:first_start].strip()
        preamble = re.sub(r'---\s*صفحة جديدة\s*---', '', preamble).strip()

        if preamble:
            clauses.append({
                "clause_id": 1,
                "clause_label": "تمهيد / مقدمة العقد",
                "clause_text": preamble
            })

        # ب) استخراج البنود المنظمة
        for idx, match in enumerate(matches):
            label = next((g for g in match.groups() if g), "").strip()
            start_pos = match.end()
            end_pos = matches[idx + 1].start() if idx + 1 < len(matches) else len(prepared_text)

            clause_body = prepared_text[start_pos:end_pos].strip()
            clause_body = re.sub(r'---\s*صفحة جديدة\s*---', '', clause_body).strip()

            if clause_body:
                clauses.append({
                    "clause_id": len(clauses) + 1,
                    "clause_label": label,
                    "clause_text": clause_body
                })

        return clauses

    paragraphs = [p.strip() for p in prepared_text.split("\n\n") if p.strip()]
    return [{"clause_id": i + 1, "clause_label": f"فقرة {i + 1}", "clause_text": p} for i, p in enumerate(paragraphs)]


# ==========================================
# التنفيذ والتخزين مباشرة دون طباعة
# ==========================================
target_input = globals().get('clean_text') or globals().get('full_extracted_text', '')

if target_input:
    clauses = segment_legal_clauses_exact(target_input)
else:
    clauses = []

## 9️⃣ Clause Segmentation — تقسيم البنود

نتعرف على أنماط ترقيم البنود القانونية الشائعة في العربية:

- `البند الأول` / `البند الثاني` ...
- `أولاً` / `ثانياً` / `ثالثاً` ...
- `المادة الأولى` / `المادة الثانية` ...
- `1-` / `2-` / `3-` ...
- `(1)` / `(2)` / `(3)` ...

إذا لم توجد أرقام واضحة، يتم استخدام تقسيم الفقرات كخطة بديلة (fallback). **لا يتم اختراع أي بند غير موجود فعليًا في النص.**


In [ ]:
import re

def fix_ocr_errors_pure_python(text):
    if not text:
        return ""

    # 1. إصلاح عناوين البنود المقسومة على سطرين أو بينها رموز/أسطر فارغة
    text = re.sub(r'(البند\s+[أ-ي]+)\s*[\r\n\s_]+(عشر)', r'\1 \2', text)
    text = re.sub(r'(المادة\s+[أ-ي]+)\s*[\r\n\s_]+(عشر)', r'\1 \2', text)

    # 2. تصحيح البنود المبتورة إملائياً
    text = re.sub(r'\bالبند\s*الرا\b', 'البند الرابع', text)
    text = re.sub(r'\bالبند\s*الرا[بع]*\b', 'البند الرابع', text)
    text = re.sub(r'\bالبند\s*السا[يئ]ع\b', 'البند السابع', text)
    text = re.sub(r'\bاليند\b', 'البند', text)

    # 3. تصحيح الرقم 2 المتحول لعلامة تنصيص " أو ؟ داخل الأقواس
    def fix_brackets(match):
        inner = match.group(1)
        inner = inner.replace('"', '2').replace('؟', '2').replace("'", '')
        return f"({inner.strip()})"

    text = re.sub(r'\(([^)\n]*[\d٠-٩"؟][^)\n]*)\)', fix_brackets, text)

    # 4. تنظيف المسافات والرموز العشوائية الخفية
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text


def segment_legal_clauses_exact(text):
    if not text or not text.strip():
        return []

    # 1. تصحيح النص كاملاً بالـ Pure Python
    prepared_text = fix_ocr_errors_pure_python(text)

    # 2. استكمال باقي خطوات التقسيم بـ Regex
    clause_regex = build_advanced_clause_pattern()
    matches = list(clause_regex.finditer(prepared_text))

    clauses = []
    if matches:
        first_start = matches[0].start()
        preamble = prepared_text[:first_start].strip()
        preamble = re.sub(r'---\s*صفحة جديدة\s*---', '', preamble).strip()

        if preamble:
            clauses.append({
                "clause_id": 1,
                "clause_label": "تمهيد / مقدمة العقد",
                "clause_text": preamble
            })

        for idx, match in enumerate(matches):
            label = next((g for g in match.groups() if g), "").strip()
            start_pos = match.end()
            end_pos = matches[idx + 1].start() if idx + 1 < len(matches) else len(prepared_text)

            clause_body = prepared_text[start_pos:end_pos].strip()
            clause_body = re.sub(r'---\s*صفحة جديدة\s*---', '', clause_body).strip()

            if clause_body:
                clauses.append({
                    "clause_id": len(clauses) + 1,
                    "clause_label": label,
                    "clause_text": clause_body
                })

        return clauses

    # الخيار البديل عند عدم العثور على أنماط بنود صريحة
    paragraphs = [p.strip() for p in prepared_text.split("\n\n") if p.strip()]
    return [{"clause_id": i + 1, "clause_label": f"فقرة {i + 1}", "clause_text": p} for i, p in enumerate(paragraphs)]

## 🔟 Quality Check — فحص الجودة

In [ ]:
def word_count(text):
    """حساب عدد الكلمات في النص بشكل دقيق مع تجنب الترقيم المنفرد."""
    if not text:
        return 0
    # استخراج الكلمات الفعلية فقط واستبعاد علامات الترقيم والرموز المنفصلة
    words = re.findall(r'\b\w+\b', text)
    return len(words)

# فحص آمن لمتغير البنود ووجود الملفات
num_clauses = len(clauses) if 'clauses' in locals() and clauses is not None else 0
file_path_exists = 'uploaded_file_path' in locals() and uploaded_file_path and os.path.exists(uploaded_file_path)

raw_words = word_count(raw_text) if 'raw_text' in locals() else 0
clean_words = word_count(clean_text) if 'clean_text' in locals() else 0

quality_report = {
    "document_name": os.path.basename(uploaded_file_path) if file_path_exists else None,
    "file_type": detect_file_type(uploaded_file_path) if file_path_exists else None,
    "extraction_method": extraction_method if 'extraction_method' in locals() else None,
    "number_of_pages": num_pages if 'num_pages' in locals() else 0,
    "word_count_raw": raw_words,
    "word_count_clean": clean_words,
    "word_retention_rate": f"{(clean_words / raw_words * 100):.1f}%" if raw_words > 0 else "0%",
    "number_of_clauses": num_clauses,
    "processed_at": datetime.now().isoformat(timespec="seconds"),
}

print("📊 تقرير الجودة واستخراج البيانات:")
for k, v in quality_report.items():
    print(f"   - {k}: {v}")

📊 تقرير الجودة واستخراج البيانات:
   - document_name: عقد بيع الدور الثانى با رقم لعمارة 15 (1).pdf
   - file_type: pdf
   - extraction_method: None
   - number_of_pages: 0
   - word_count_raw: 0
   - word_count_clean: 876
   - word_retention_rate: 0%
   - number_of_clauses: 9
   - processed_at: 2026-09-09T08:31:54


## 1️⃣1️⃣ Preview — معاينة النتائج

In [ ]:
print("=" * 60)
print("📄 CLEAN TEXT (كامل نص العقد)")
print("=" * 60)
print(clean_text if 'clean_text' in locals() and clean_text else "(فارغ)")

print("\n" + "=" * 60)

# التأكد الآمن من وجود قائمة البنود وتجنب الأخطاء
all_clauses = clauses if 'clauses' in locals() and clauses is not None else []

print(f"📑 CLAUSES (عرض كافة البنود - إجمالي {len(all_clauses)} بند)")
print("=" * 60)

if all_clauses:
    for c in all_clauses:
        clause_id = c.get("clause_id", "N/A")
        label = f"[{c.get('clause_label')}]" if c.get("clause_label") else "[بدون عنوان]"
        text_content = c.get("clause_text", "").strip()

        print(f"📌 البند {clause_id}: {label}\n{text_content}\n")
        print("-" * 40)
else:
    print("(لم يتم تقسيم البنود بعد أو القائمة فارغة)")

📄 CLEAN TEXT (كامل نص العقد)
عقد بيع نهائى للشقة رقم (٠١") بالدور الثانى
بالعمارة رقم (١5) مدينة زهور المعادى (أبراج بدر)
إنه فى يوم الثلاثاء الموافق 5١75/10/١4
حرر هذا العقد بين كلا من:
السيد /

ومقرههاعمارةرقم (١٠١) مدينة زهور المعادى (ابراج بدر) - الطريق
الدائرى - بجوار كارفور المعادى.

(طرف أول بائع)

ثانياً: السيد/

ومقيم فى /
بطاقة رقم /
(طرف ثان مشترى)
وبعد أن أقر الطرفان بأهليتهمما وصلاحيتهما للتعاقد وخلو الطرفان من الموانع
القانونية إتفقا على ما يأتى: -
تمهيد

يمتدك الطرف الأول قطعة الأرض رقم (3) الواقعه بأرض الأستثمار بالقطامية - الطريق
الدائرى - المعادى الجديدة - محافظة القاهرة وذلك بموجب قرار التخصيص الصادر من
محافظة القاهرة - البساتين برقم ٠ فى ١119/8/2 ومحضر الاستلام من مديرية
مساحة القاهرة فى ١119/8/55 والصصادر بهاقرار التقسيم رقم VVY بتاريخ
1ل ممديرية الإسكان والمرافق طبقا للرسومات الهندسية المعتمدة والمنشورة
بالوقائع المصرية (العدد ١545 فى) ١1913/٠١/5١ والمسجل والمشهر عنه برقم ٠١5*
لسنة ٠٠٠١ شهر عقارى جنوب القاهرة بتاريخ ٠٠٠١/5/75 ورغبة من الطرف الثانى
فى تملك شقة بمدي